# ShotLab — YOLO ball detector training (Kaggle GPU)

One-button cloud training. Reproduces the local run
`train_ball.py --data dataset_ball_human.yaml --base ball_orange/best.pt --freeze 10`
on a free NVIDIA GPU (stock CUDA — the most-tested path, no AMD/DirectML quirks).

**Before running:** Settings -> Accelerator -> **GPU T4 x1** (or P100).
Attach the **shotlab-ball-dataset** dataset (the zip from `tools/pack_kaggle_dataset.py`).

Then Run All. Download `best.pt` + `best.onnx` from the **Output** tab when done,
and verify locally per `process/KAGGLE_TRAINING.md` (compare to the CPU-trained `ball_human`).


## 1. Environment + GPU


In [ ]:
import subprocess, sys
# pin ultralytics to the local version for reproducibility; keep Kaggle's CUDA torch
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.104'], check=True)
import torch
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU T4 x1 in Settings'
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))

## 2. Locate the dataset + write the Kaggle-path data.yaml
Mirrors `dataset_ball_human.yaml`: the two REAL dirs (close set + human far set), no synthetic aug.


In [ ]:
import glob, os, yaml, pathlib
# find the attached dataset at ANY depth under /kaggle/input (Kaggle may nest
# it under datasets/<user>/<slug>/ depending on how it was added)
hits = glob.glob('/kaggle/input/**/dataset_ball_labeled', recursive=True)
assert hits, 'shotlab dataset not attached — Add Input -> your uploaded dataset'
BASE = str(pathlib.Path(hits[0]).parent)
print('dataset root:', BASE)
data = {
  'train': [f'{BASE}/dataset_ball/images/train', f'{BASE}/dataset_ball_labeled/images/train'],
  'val':   [f'{BASE}/dataset_ball/images/val',   f'{BASE}/dataset_ball_labeled/images/val'],
  'nc': 1, 'names': ['ball'],
}
YAML = '/kaggle/working/dataset_kaggle.yaml'
yaml.safe_dump(data, open(YAML,'w'))
BASE_WEIGHTS = f'{BASE}/ball_orange_best.pt'
assert os.path.exists(BASE_WEIGHTS), BASE_WEIGHTS
for k in ('train','val'):
    for p in data[k]:
        n = len(glob.glob(p+'/*.jpg'))
        print(f'{k}: {n:5d}  {p}')
print('base weights:', BASE_WEIGHTS)

## 3. Train
yolo11n, imgsz 1280, freeze-10 backbone, 40 epochs. Same augmentation as `train_ball.py`.

`batch=8` is safe on a 16 GB T4 at 1280; bump to 16 on a P100 if you like.


In [ ]:
from ultralytics import YOLO
model = YOLO(BASE_WEIGHTS)
model.train(
    data=YAML,
    epochs=40, imgsz=1280, batch=8, freeze=10,
    device=0, amp=True, patience=15,
    mosaic=1.0, close_mosaic=10,
    hsv_h=0.02, hsv_s=0.5, hsv_v=0.4,
    fliplr=0.5, scale=0.5, translate=0.1, degrees=5.0,
    project='/kaggle/working/runs', name='ball_gpu_kaggle', exist_ok=True,
    verbose=True,
)

## 4. Validate, sanity-gate, export ONNX
The `mAP50 > 0.1` assert is the cloud analogue of the DirectML box/dfl-loss check:
a backend that 'trains' but learns nothing (like DirectML did) fails here.


In [ ]:
m = model.val(data=YAML)
print(f'mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')
assert m.box.map50 > 0.1, 'model learned nothing — do NOT ship this checkpoint'

import shutil, glob
best = sorted(glob.glob('/kaggle/working/runs/**/weights/best.pt', recursive=True))[-1]
onnx = model.export(format='onnx', imgsz=1280)   # for local DirectML inference
shutil.copy(best, '/kaggle/working/best.pt')
shutil.copy(onnx, '/kaggle/working/best.onnx')
print('\nDownload from the Output tab: best.pt  best.onnx')
print('Then verify locally vs ball_human — see process/KAGGLE_TRAINING.md')